# Norvig-style autocorrect

Peter Norvig has a famous ~30-line spelling corrector that's worth understanding before reaching for anything fancier. The idea: build a word frequency table, generate every string at edit-distance 1 or 2 from the input, filter to known words, return the highest-frequency candidate.

I'm using the NLTK `words` corpus as a vocabulary, weighted by counts from a built-in text. If NLTK's corpora aren't downloaded I fall back to Python's own list (less ideal, no frequency signal).

In [1]:
# !pip install nltk -q

In [2]:
import re
from collections import Counter

# Try NLTK first; fall back to a built-in word list if not available.
WORDS = None
try:
    import nltk
    try:
        from nltk.corpus import words as nltk_words
        from nltk.corpus import brown
        _ = nltk_words.words()  # touches it to confirm download
        _ = brown.words()
    except LookupError:
        nltk.download("words", quiet=True)
        nltk.download("brown", quiet=True)
        from nltk.corpus import words as nltk_words
        from nltk.corpus import brown

    # use Brown for frequency, words corpus to constrain vocabulary
    vocab = set(w.lower() for w in nltk_words.words())
    WORDS = Counter(w.lower() for w in brown.words() if w.lower() in vocab)
    print("vocab size:", len(vocab), "  total tokens in freq table:", sum(WORDS.values()))
except Exception as e:
    print("NLTK route failed, using a tiny built-in word list:", repr(e)[:120])
    builtin = ["the","be","to","of","and","a","in","that","have","for","not","on",
               "with","he","as","you","do","at","this","but","his","by","from",
               "they","we","say","her","she","or","an","will","my","one","all",
               "would","there","their","what","so","up","out","if","about","who",
               "spelling","correction","example","python","computer","science"]
    WORDS = Counter(builtin)

vocab size: 234377   total tokens in freq table: 870128


In [3]:
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def P(word, N=None):
    N = N or sum(WORDS.values())
    return WORDS[word] / N if N else 0

def known(words_iter):
    return {w for w in words_iter if w in WORDS}

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes    = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces   = [L + c + R[1:] for L, R in splits if R for c in LETTERS]
    inserts    = [L + c + R     for L, R in splits     for c in LETTERS]
    return set(deletes + transposes + replaces + inserts)

def edits2(word):
    # be careful: this set is large. don't materialize for long words without bound.
    return {e2 for e1 in edits1(word) for e2 in edits1(e1)}

def candidates(word):
    return (known([word]) or known(edits1(word)) or known(edits2(word)) or {word})

def correct(word):
    return max(candidates(word), key=P)

## Demo on a handful of misspellings

In [4]:
tests = [
    "speling",      # standard spelling typo
    "korrect",      # phonetic
    "comuter",      # missing char
    "thier",        # transpose
    "recieve",      # ei/ie classic
    "alot",         # not really a typo but commonly autocorrected
    "fhqwhgads",    # nonsense — should pass through
]

for t in tests:
    c = correct(t)
    flag = "(unchanged)" if c == t else f"-> {c}"
    print(f"{t:>10}  {flag}")

   speling  -> spelling
   korrect  -> correct
   comuter  -> computer
     thier  -> their
   recieve  -> receive
      alot  -> lot


 fhqwhgads  (unchanged)


## On a small sentence

In [5]:
def correct_sentence(s):
    tokens = re.findall(r"[A-Za-z']+|[^A-Za-z']", s)
    out = []
    for tok in tokens:
        if tok[:1].isalpha():
            corrected = correct(tok.lower())
            # preserve original casing for the first letter, roughly
            if tok[:1].isupper():
                corrected = corrected[:1].upper() + corrected[1:]
            out.append(corrected)
        else:
            out.append(tok)
    return "".join(out)

print(correct_sentence("I beleive teh quik brown fox jumpes over a lasy dog."))

I believe the quick brown fox jumper over a last dog.


## Notes

What Norvig's autocorrect gets right: it's small enough to read in one sitting, and on simple single-token typos it does pretty well.

What it doesn't do (and these matter):

- It has no notion of context. "their" vs "there" vs "they're" all hash to themselves; the corrector won't help you between them.
- The frequency model is unigram. Real autocorrect on a phone uses at minimum a bigram or trigram model over recently typed tokens.
- Edits beyond distance 2 are not generated, which means long compound typos sail past unchanged.

The standard production upgrade is a noisy channel model with a character-level edit model (SymSpell is a much faster implementation) plus a language model for context. Out of scope here.